### Grid Search CV

In [51]:
import pandas as pd
import seaborn as sns

In [52]:
df = sns.load_dataset('iris')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


In [53]:
df.drop_duplicates()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [54]:
from sklearn.model_selection import train_test_split
X = df.drop('species',axis=1)
y = df['species']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [55]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=30 , weights='distance' , algorithm='auto' ,metric='minkowski')
knn_model.fit(X_train , y_train)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",30
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](3,)","['setosa','versicolor','virginica']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [56]:
y_predict = knn_model.predict(X_test)

In [57]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test , y_predict)

0.9666666666666667

First, we choose a model and define its hyperparameters with different values. Then GridSearchCV creates all possible combinations of these hyperparameters. For each combination, it applies cross validation on the dataset by splitting it into folds and calculates the average accuracy. After testing all combinations, it selects the best hyperparameters based on the highest score and returns the best model.

In [58]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=knn_model,
    param_grid={
        'n_neighbors': [1,2,3,4,5,6,7,8,9],
        'weights': ['uniform', 'distance'],
        'algorithm': ['ball_tree', 'kd_tree', 'brute'],
        'metric': ['euclidean', 'manhattan', 'minkowski']
    },
    cv=5,
    scoring='accuracy'
)

grid.fit(X , y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsCla...ts='distance')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'algorithm': ['ball_tree', 'kd_tree', ...], 'metric': ['euclidean', 'manhattan', ...], 'n_neighbors': [1, 2, ...], 'weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Suppor

In [59]:
print(grid.best_params_)   
print(grid.best_score_)   


{'algorithm': 'ball_tree', 'metric': 'euclidean', 'n_neighbors': 6, 'weights': 'uniform'}
0.9800000000000001


In [60]:
print(grid.cv_results_)
dataF = pd.DataFrame(data=grid.cv_results_)
dataF

{'mean_fit_time': array([0.00560083, 0.00351825, 0.00300474, 0.00319982, 0.00326519,
       0.00319963, 0.00339837, 0.00319614, 0.0032001 , 0.00340009,
       0.00299959, 0.00339999, 0.0032002 , 0.00319996, 0.00320005,
       0.00299988, 0.00339999, 0.00300045, 0.00359674, 0.00299997,
       0.00319724, 0.00299997, 0.00319977, 0.00319977, 0.00339975,
       0.00340018, 0.00339952, 0.0032002 , 0.00312095, 0.00312862,
       0.00300064, 0.00299716, 0.00359988, 0.00305972, 0.00299997,
       0.00320005, 0.00280013, 0.00319991, 0.00279994, 0.00315604,
       0.0032001 , 0.00319986, 0.00320005, 0.00299997, 0.0027987 ,
       0.00280042, 0.00299983, 0.00339999, 0.00280004, 0.00320001,
       0.00320683, 0.00299683, 0.00524278, 0.00469794, 0.0045517 ,
       0.00339961, 0.0030004 , 0.00300026, 0.00319777, 0.00300002,
       0.00300145, 0.00373235, 0.00350962, 0.00354848, 0.00312328,
       0.00349817, 0.00329099, 0.00333905, 0.00326524, 0.00321746,
       0.0033453 , 0.00349669, 0.00307612, 0

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_algorithm,param_metric,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.005601,0.001019,0.009601,0.001960,ball_tree,euclidean,1,uniform,"{'algorithm': 'ball_tree', 'metric': 'euclidea...",0.966667,0.966667,0.933333,0.933333,1.0,0.960000,0.024944,107
1,0.003518,0.001037,0.004200,0.000400,ball_tree,euclidean,1,distance,"{'algorithm': 'ball_tree', 'metric': 'euclidea...",0.966667,0.966667,0.933333,0.933333,1.0,0.960000,0.024944,107
2,0.003005,0.000009,0.079487,0.148975,ball_tree,euclidean,2,uniform,"{'algorithm': 'ball_tree', 'metric': 'euclidea...",0.966667,0.933333,0.933333,0.900000,1.0,0.946667,0.033993,154
3,0.003200,0.000400,0.004200,0.000400,ball_tree,euclidean,2,distance,"{'algorithm': 'ball_tree', 'metric': 'euclidea...",0.966667,0.966667,0.933333,0.933333,1.0,0.960000,0.024944,107
4,0.003265,0.000395,0.005335,0.000429,ball_tree,euclidean,3,uniform,"{'algorithm': 'ball_tree', 'metric': 'euclidea...",0.966667,0.966667,0.933333,0.966667,1.0,0.966667,0.021082,57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,0.003207,0.000399,0.008518,0.000500,brute,minkowski,7,distance,"{'algorithm': 'brute', 'metric': 'minkowski', ...",0.966667,1.000000,0.966667,0.966667,1.0,0.980000,0.016330,1
158,0.003719,0.000879,0.010145,0.001950,brute,minkowski,8,uniform,"{'algorithm': 'brute', 'metric': 'minkowski', ...",0.966667,1.000000,0.933333,0.933333,1.0,0.966667,0.029814,57
159,0.004000,0.000632,0.009144,0.000912,brute,minkowski,8,distance,"{'algorithm': 'brute', 'metric': 'minkowski', ...",0.966667,1.000000,0.966667,0.966667,1.0,0.980000,0.016330,1
160,0.002800,0.000400,0.008007,0.000418,brute,minkowski,9,uniform,"{'algorithm': 'brute', 'metric': 'minkowski', ...",0.966667,1.000000,0.966667,0.966667,1.0,0.980000,0.016330,1


In [65]:
from sklearn.model_selection import RandomizedSearchCV

grid_r = RandomizedSearchCV(
    estimator=knn_model,
    param_distributions={
        'n_neighbors': [1,2,3,4,5,6,7,8,9],
        'weights': ['uniform', 'distance'],
        'algorithm': ['ball_tree', 'kd_tree', 'brute'],
        'metric': ['euclidean', 'manhattan', 'minkowski']
    },
    n_iter=5,
    cv=5,
    scoring='accuracy'
)

grid_r.fit(X , y)


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsCla...ts='distance')
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'algorithm': ['ball_tree', 'kd_tree', ...], 'metric': ['euclidean', 'manhattan', ...], 'n_neighbors': [1, 2, ...], 'weights': ['uniform', 'distance']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sp

In [67]:
print(grid_r.best_score_)
print(grid_r.best_params_)

0.9733333333333334
{'weights': 'uniform', 'n_neighbors': 4, 'metric': 'minkowski', 'algorithm': 'ball_tree'}
